In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf

In [4]:
spark=SparkSession.builder.appName("UDF").getOrCreate()

In [5]:
data = [
    (1, "  alice  ", 22, 55000),
    (2, "BOB", 17, 30000),
    (3, "  charlie", 25, 75000),
    (4, "DAVID ", 30, 45000),
    (5, " eve ", 19, 60000)
]

In [6]:
columns=["id" , "name" , "age" , "salary"]

In [7]:
df=spark.createDataFrame(data , columns)

In [8]:
df.show()

+---+---------+---+------+
| id|     name|age|salary|
+---+---------+---+------+
|  1|  alice  | 22| 55000|
|  2|      BOB| 17| 30000|
|  3|  charlie| 25| 75000|
|  4|   DAVID | 30| 45000|
|  5|     eve | 19| 60000|
+---+---------+---+------+



## UDF

In [9]:
def make_upper(name):
  return name.upper()

In [10]:
from pyspark.sql.functions import udf

In [11]:
from pyspark.sql.functions import StringType

In [12]:
upper_udf=udf(make_upper , StringType())

In [13]:
df_udf=df.withColumn(
    "name_upper" , upper_udf(sf.col("name"))
)

In [14]:
df_udf.show()

+---+---------+---+------+----------+
| id|     name|age|salary|name_upper|
+---+---------+---+------+----------+
|  1|  alice  | 22| 55000|   ALICE  |
|  2|      BOB| 17| 30000|       BOB|
|  3|  charlie| 25| 75000|   CHARLIE|
|  4|   DAVID | 30| 45000|    DAVID |
|  5|     eve | 19| 60000|      EVE |
+---+---------+---+------+----------+



##BUILT IN FUNCTIONS

In [15]:
df_builtin=df.withColumn(
    "name_upper" , sf.upper(sf.col("name"))
)


In [16]:
df_builtin.show()

+---+---------+---+------+----------+
| id|     name|age|salary|name_upper|
+---+---------+---+------+----------+
|  1|  alice  | 22| 55000|   ALICE  |
|  2|      BOB| 17| 30000|       BOB|
|  3|  charlie| 25| 75000|   CHARLIE|
|  4|   DAVID | 30| 45000|    DAVID |
|  5|     eve | 19| 60000|      EVE |
+---+---------+---+------+----------+



## RESUABLE TRASFORMATION FUNCTIONS

In [17]:
def clean_names(df):
  return df.withColumn(
      "name" ,
      sf.upper(sf.trim(sf.col("name")))
  )

In [18]:
def filter_adults(df):
  return df.filter(
      sf.col("age") >=20
  )

In [19]:
def add_salary_category(df):
  return df.withColumn(
      "salary_category" ,
      sf.when(sf.col("salary") >= 50000 , "high")
      .when(sf.col("salary") >= 40000 , "Medium")
      .otherwise("Low")
  )

In [20]:
def select_required_columns(df):
  return df.select("id" , "name" , "age" , "salary_category")

In [21]:
def transformation(df):
  df=clean_names(df)
  df=filter_adults(df)
  df=add_salary_category(df)
  df=select_required_columns(df)
  return df

In [22]:
result=transformation(df)

In [23]:
result.show()

+---+-------+---+---------------+
| id|   name|age|salary_category|
+---+-------+---+---------------+
|  1|  ALICE| 22|           high|
|  3|CHARLIE| 25|           high|
|  4|  DAVID| 30|         Medium|
+---+-------+---+---------------+



## Testing

In [25]:
assert "salary_category" in result.columns

In [28]:
assert result.filter(
    sf.col("age") < 18
).count() == 0